# Exercise 01 — Introduction to Google Colab

**MSc Finance · Investments · FHNW**

Welcome to the first exercise. Its main goal is to get you comfortable with **Google Colab**,
the environment we will use for all empirical exercises in this course. Along the way you
will already do some real finance: loading historical market data and measuring return and risk.


## What is Google Colab — and how does this work?

**Google Colab** is a free service that runs Python code in your browser, on Google's
servers. You need nothing installed on your own computer — just a (free) Google account.

A Colab document is a **notebook**: a sequence of *cells*. There are two kinds:
- **Text cells** (like this one) explain what is going on.
- **Code cells** contain Python code. You run a cell by clicking it and pressing
  **`Shift + Enter`** (or the ▶ button on its left). Run the cells **from top to bottom** —
  later cells rely on earlier ones.

**Where does this notebook come from?** It is stored in a public **GitHub** repository — an
online folder for course files. The Colab link on Moodle tells Colab to open the notebook
straight from GitHub, so everyone always gets the latest version. The dataset is loaded the
same way, directly from the repository (you will see this below).

**Can I change it and keep my work?** Yes — and you should experiment freely. You are editing
a *temporary copy*; your changes are **not** saved back to GitHub. To keep your work, use the
menu **File → Save a copy in Drive**. This puts your own editable copy in your Google Drive.
You can also download it via **File → Download → Download .ipynb**.


## 1 · Setup

Python's power comes from **libraries** — collections of ready-made tools. We `import` the
three we need and give them their conventional short names:
- `numpy` (`np`) — numerical operations,
- `pandas` (`pd`) — working with tables of data,
- `matplotlib.pyplot` (`plt`) — plotting.

The `plt.rcParams.update(...)` block just sets a few visual defaults so our charts look tidy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# FHNW colour palette (consistent with the lecture slides and dashboards)
FHNW = {
    "navy":   "#1B3A5C",
    "blue":   "#0057A4",
    "green":  "#2E7D32",
    "red":    "#C8102E",
    "orange": "#EA8700",
    "yellow": "#FFD500",
}
ASSET_COLORS = {
    "S&P 500 (includes dividends)": FHNW["blue"],
    "US T. Bond": FHNW["green"],
    "3-month T.Bill": FHNW["orange"],
}

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 2 · Import the data

The data is an Excel file in the course repository. The command `pd.read_excel(URL)` reads
it **directly from the web** — there is nothing to download by hand. We then use the column
`Year` as the table's row index, and look at the first few rows with `.head()`.

These are historical **total-return indices**: each series starts at 100 in 1927 and grows
as if all income (dividends, coupons) were reinvested. The file contains several asset
classes; in the next step we will narrow down to the three we study here.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/KroeTiA/Investments/main/Exercise_01/data/histretSP_investments.xls"
SHEET = "Total Return Index_Damadoran"

# Read the Excel sheet straight from GitHub
raw = pd.read_excel(DATA_URL, sheet_name=SHEET)

# Tidy column names (remove stray spaces) and use Year as the row index
raw.columns = [str(c).strip() for c in raw.columns]
idx_all = raw.set_index("Year")

# Drop the trailing source-note column if present
idx_all = idx_all.loc[:, ~idx_all.columns.str.contains("Source", case=False, na=False)]

print(f"Loaded {idx_all.shape[0]} annual observations for {idx_all.shape[1]} asset classes:")
print(list(idx_all.columns))
idx_all.head()

### Select the three asset classes we study

The file holds several asset classes. To make sure we always analyse **the same three**, we
explicitly keep only:
- **S&P 500** — broad US stock market (with dividends),
- **US T. Bond** — long-term US government bonds,
- **3-month T.Bill** — short-term government bills (our risk-free asset).

Selecting columns by name from a DataFrame is done with a list of column names in square
brackets: `idx_all[[...]]`.

In [ ]:
ASSETS = ['S&P 500 (includes dividends)', 'US T. Bond', '3-month T.Bill']

idx = idx_all[ASSETS]          # keep only our three asset classes

print(f"Now analysing {idx.shape[1]} asset classes: {list(idx.columns)}")
idx.head()

### Total-return indices over time

A quick plot to see the data. We use a **logarithmic** y-axis: on a log scale, equal vertical
distances correspond to equal *percentage* changes, which is the natural way to compare
long-run growth of assets that differ hugely in scale.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for col in idx.columns:
    ax.plot(idx.index, idx[col], label=col, color=ASSET_COLORS.get(col), lw=1.8)
ax.set_yscale("log")
ax.set_ylabel("Index level (log scale, base = 100 in 1927)")
ax.set_xlabel("Year")
ax.set_title("Historical Total-Return Indices, 1927–2025")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 3 · From indices to returns

An index level on its own is hard to interpret. What we care about is the **return**: the
relative change from one year to the next,

$$ R_t = \frac{I_t}{I_{t-1}} - 1 $$

### 3.1 · By hand — one asset

Let us first compute this manually for the S&P 500, to see exactly what the formula does.
The idea: take the index series, and divide each value by the **previous year's** value.
In pandas, `.shift(1)` moves a series down by one row, so it lines up each year with the
*prior* year. Dividing the two series and subtracting 1 gives the annual return.

In [ ]:
# Take one asset's index series
asset = idx.columns[0]
level = idx[asset]

# The lagged series: each row now holds the PREVIOUS year's value
level_lagged = level.shift(1)

# Return = this year's level / last year's level - 1
ret_manual = level / level_lagged - 1

# The first year has no predecessor -> NaN; drop it
ret_manual = ret_manual.dropna()

print(f"Asset: {asset}")
ret_manual.head()

### 3.2 · The shortcut — `pct_change()`  🛠 *Your turn*

Doing this column by column would be tedious. pandas has a one-line method that computes the
relative change for **every asset at once**: `DataFrame.pct_change()`.

**Task:** Compute a table `rets` of annual returns from `idx` using `pct_change()`, and drop
the first (empty) row with `.dropna()`. This is a good moment to try editing a code cell
yourself.

In [ ]:
# 🛠 Your turn: compute annual returns for all assets and store them in `rets`
# rets = ...

# --- check your result (uncomment once you have `rets`) ---
# print(rets.shape)      # expect (98, 3)
# rets.head()

<details>
<summary>💡 Click for the solution</summary>

```python
rets = idx.pct_change().dropna()
rets.head()
```

The first column of `rets` is identical to the `ret_manual` we computed by hand above —
`pct_change()` simply does the same calculation for every asset at once.
</details>

## 4 · Distribution of returns

This cell draws a **histogram** for each asset: it counts how often returns fall into each
range. All panels share the **same x-axis and the same bins**, so the differences in *spread*
are directly comparable — a narrow distribution (T-Bills) means low risk, a wide one
(stocks) means high risk. The dashed line marks the mean.

In [ ]:
n = idx.shape[1]
fig_dist, axes = plt.subplots(1, n, figsize=(12, 3.6), sharex=True, sharey=True)

# Common bin edges across the three assets -> identical x-axis and bar widths
lo, hi = rets.min().min(), rets.max().max()
bins = np.linspace(lo, hi, 31)

for ax, col in zip(axes, rets.columns):
    ax.hist(rets[col], bins=bins, color=ASSET_COLORS.get(col), alpha=0.85, edgecolor="white")
    ax.axvline(rets[col].mean(), color=FHNW["navy"], ls="--", lw=1.3)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Annual return")
axes[0].set_ylabel("Frequency")
fig_dist.suptitle("Distribution of Annual Returns (common scale; dashed line = mean)", y=1.03)
fig_dist.tight_layout()
fig_dist.savefig("distribution.png", dpi=300, bbox_inches="tight")  # saved here so the export matches exactly
plt.show()

## 5 · Descriptive statistics: mean and standard deviation

We focus on the two key descriptive statistics: the **mean** (average return) and the
**standard deviation** (a measure of risk). Further statistics follow in Exercise 02.

### 5.1 · By hand — the S&P 500

Before using built-in functions, let us compute both quantities manually for the first asset,
so the formulas are transparent.

The **arithmetic mean** of $n$ returns is

$$ \bar{R} = \frac{1}{n} \sum_{t=1}^{n} R_t $$

The **sample variance** is the average squared deviation from the mean (dividing by $n-1$):

$$ s^2 = \frac{1}{n-1} \sum_{t=1}^{n} (R_t - \bar{R})^2 $$

and the **standard deviation** is its square root, $s = \sqrt{s^2}$.

In [ ]:
# Pick the first asset
asset = rets.columns[0]
x = rets[asset]
n = len(x)

# --- Mean: sum of returns divided by the number of observations
mean_manual = x.sum() / n

# --- Variance: average squared deviation from the mean, using (n - 1)
squared_devs = (x - mean_manual) ** 2
var_manual = squared_devs.sum() / (n - 1)

# --- Standard deviation: square root of the variance
std_manual = var_manual ** 0.5

print(f"Asset: {asset}")
print(f"Observations (n):     {n}")
print(f"Arithmetic mean:      {mean_manual:.4f}  ({mean_manual:.2%})")
print(f"Sample variance:      {var_manual:.4f}")
print(f"Standard deviation:   {std_manual:.4f}  ({std_manual:.2%})")

### 5.2 · The shortcut — built-in functions

pandas provides these directly. `.mean()`, `.var()` and `.std()` use the sample convention
($n-1$) by default, so they match our manual calculation exactly.

In [ ]:
print(f"Mean:   {x.mean():.4f}")
print(f"Var:    {x.var():.4f}")     # ddof=1 (n-1) by default
print(f"Std:    {x.std():.4f}")     # ddof=1 (n-1) by default

# Confirm they agree with the manual computation
assert np.isclose(x.mean(), mean_manual)
assert np.isclose(x.var(),  var_manual)
assert np.isclose(x.std(),  std_manual)
print("\n✓ Manual and built-in results agree.")

### 5.3 · All assets at once

Applied to the whole table, the same methods return one value per asset, so we can compare
mean return and risk across all three asset classes.

In [ ]:
stats_df = pd.DataFrame({
    "Mean": rets.mean(),
    "Variance": rets.var(),
    "Std. deviation": rets.std(),
})
stats_df.style.format({
    "Mean": "{:.2%}", "Variance": "{:.4f}", "Std. deviation": "{:.2%}",
})

## 6 · Export your results

Often you want to take a figure or a table out of Colab — for a report, or to keep a record.
The distribution figure above was already saved to a file. The cell below saves the
statistics table (Excel), bundles it together with the figure into a single ZIP, and
downloads it to your computer. (In Firefox/Safari you may need to allow pop-ups.)

Remember: to keep the **whole notebook** with your own edits, use **File → Save a copy in
Drive**.

In [ ]:
# The distribution figure was already saved as distribution.png in section 4.
# Here we save the statistics table and bundle everything for download.

with pd.ExcelWriter("descriptive_statistics.xlsx") as w:
    stats_df.to_excel(w, sheet_name="Descriptive")
    rets.to_excel(w, sheet_name="Returns")

# Bundle the figure and the table into one ZIP for download
import zipfile
with zipfile.ZipFile("exercise_01_results.zip", "w") as z:
    z.write("distribution.png")
    z.write("descriptive_statistics.xlsx")

try:
    from google.colab import files
    files.download("exercise_01_results.zip")
except ImportError:
    print("Not running in Colab — files saved to the working directory.")